# 04 · Generate Evaluation Outputs and Blinded Pairs
**PA2 Task 6 (Sec. 12).** Attach the saved outputs of notebooks **01** and **02** (Add Input → Your Work → Notebooks). Runtime ~20-40 min.

| Comparison | Eval set | Judge mode |
|---|---|---|
| DPO-HH vs SFT | `eval_sets/hh_rlhf_eval_300_seed42.json` (provided synthetic helpful/harmless set) | safety |
| DPO-UF vs SFT | `eval_sets/alpaca_eval_300_seed42.json` (300 prompts, seed 42) | general |

Identical decoding for every model: greedy, 512 new tokens, repetition penalty 1.1.

## Step 0 · Copy the code into Kaggle's working folder  (PA2 Task 1, Sec. 7 + Sec. 5.4)
Notebook settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified Kaggle account).

The code comes from your uploaded Kaggle Dataset (the `small-model-alignment.zip`). Attach it with **+ Add Input → Datasets → Your Datasets**. This cell finds it automatically, so there are no paths to edit.

In [ ]:
# =============================================================================
# Step 0: copy the code from the attached Kaggle Dataset into /kaggle/working
#   /kaggle/working is the ONLY folder Kaggle keeps when you "Save Version" (PA2 Sec. 5.5)
# =============================================================================
import os, shutil, glob

REPO_DIR = "/kaggle/working/small-model-alignment"

def find_code_dataset():
    # Locate the uploaded code dataset anywhere under /kaggle/input.
    # Attached notebook OUTPUTS also contain a copy of the repo, but those copies
    # have an outputs/ folder; the clean code dataset does not, so prefer it.
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)]
    clean = [h for h in hits if not os.path.isdir(os.path.join(h, "outputs"))]
    if not (clean or hits):
        raise FileNotFoundError("Code dataset not found: + Add Input -> Datasets -> Your Datasets -> your code dataset")
    return (clean or hits)[0]

if not os.path.exists(os.path.join(REPO_DIR, "submission_adapter.py")):
    src = find_code_dataset()
    print("copying code from", src)
    shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
    !chmod -R u+w {REPO_DIR}   # /kaggle/input is read-only; make the copy writable
%cd {REPO_DIR}

# Memory/perf environment for the T4s, inherited by every `!python` / `!torchrun` below:
#   expandable_segments -> less CUDA fragmentation with dynamic (per-batch) padding
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def sh(cmd):
    # Run a shell command, stream its output here, and STOP the notebook if it fails
    # (so a crashed torchrun does not silently continue during Save & Run All).
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"command failed: {cmd[:150]}")
!ls

## Step 1 · Install dependencies without touching PyTorch  (PA2 Sec. 5.2)
Kaggle's preinstalled PyTorch already matches the T4 CUDA runtime, so we never reinstall it. `torchao` is removed because old Kaggle builds break PEFT/LoRA loading.

In [ ]:
# =============================================================================
# Step 1: install the pinned libraries (exactly the manual's safe pattern)
# =============================================================================
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

## Step 2 · Verify the two T4 GPUs and print versions  (PA2 Task 1 items 3-4)
Expect two *Tesla T4* rows with ~15 GiB each. T4 = Turing (sm_75): fast **fp16** tensor cores, **no bf16**, no FlashAttention-2 → the code uses fp16 autocast + PyTorch SDPA attention.

In [ ]:
# =============================================================================
# Step 2: GPU + package check
# =============================================================================
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs visible:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Set Accelerator = GPU T4 x2 in the notebook settings"
NPROC = torch.cuda.device_count()

## Step 3 · Locate checkpoints and regenerate the fixed AlpacaEval subset

In [ ]:
# =============================================================================
# Helper: find a checkpoint inside attached Kaggle inputs (PA2 Sec. 5.4 items 5-6)
#   A saved notebook version attached as input appears under /kaggle/input/...
# =============================================================================
def find_ckpt(rel):
    """rel like 'outputs/sft/best' -> absolute folder that contains config.json"""
    local = os.path.join(REPO_DIR, rel)
    if os.path.exists(os.path.join(local, "config.json")):
        return local
    hits = sorted(glob.glob(f"/kaggle/input/**/{rel}/config.json", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{rel} not found - did you attach the previous notebook's output (Add Input)?")
    return os.path.dirname(hits[0])

CKPT = {"sft": find_ckpt("outputs/sft/best"), "dpo_hh": find_ckpt("outputs/dpo_hh/best"), "dpo_uf": find_ckpt("outputs/dpo_uf/best")}
import json
print(json.dumps(CKPT, indent=1))
!python scripts/prepare_data.py --only alpaca   # same 300 prompts, seed 42
# Bring the base-model generations from notebook 01 along (for your own analysis)
for f in glob.glob("/kaggle/input/**/outputs/generations/base_*.json", recursive=True):
    os.makedirs("outputs/generations", exist_ok=True); shutil.copy(f, "outputs/generations/")

## Step 4 · Sanity check on 3 prompts per model  (Sec. 12, first paragraph)
Confirm path, tokenizer, template, EOS behaviour and length look right before the full run.

In [ ]:
# =============================================================================
# Step 4: sanity check
# =============================================================================
for name, path in CKPT.items():
    print("#" * 30, name)
    !python scripts/generate.py --model_path {path} --generator {name} --limit 3 --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/_sanity_{name}.json

## Step 5 · Full generation, two models at a time (one per T4)

In [ ]:
# =============================================================================
# Step 5: 4 generation jobs = 2 rounds x 2 GPUs
# =============================================================================
import subprocess, time
os.makedirs("outputs/generations", exist_ok=True)
HH, AE = "eval_sets/hh_rlhf_eval_300_seed42.json", "eval_sets/alpaca_eval_300_seed42.json"

def launch(gpu, name, eval_set, out):
    cmd = (f"CUDA_VISIBLE_DEVICES={gpu} python scripts/generate.py --model_path {CKPT[name]} --generator {name} "
           f"--eval_set {eval_set} --output {out} --max_new_tokens 512 --repetition_penalty 1.1")
    return subprocess.Popen(cmd, shell=True, stdout=open(out + ".log", "w"), stderr=subprocess.STDOUT)

t = time.time()
for round_jobs in [[("sft", HH, "sft_hh"), ("dpo_hh", HH, "dpo_hh_hh")],
                   [("sft", AE, "sft_alpaca"), ("dpo_uf", AE, "dpo_uf_alpaca")]]:
    procs = [launch(g, n, e, f"outputs/generations/{o}.json") for g, (n, e, o) in enumerate(round_jobs)]
    for p in procs: p.wait(); print("exit", p.returncode)
print(f"{(time.time()-t)/60:.1f} min")
!tail -n 2 outputs/generations/*.log

## Step 6 · Build blinded pairwise files  (Sec. 12)
Order of the two responses is randomised per pair (seed 42); which model is A or B is stored only in hidden `meta`, never shown to the judge (same practice as RA2 App. C.2).

In [ ]:
# =============================================================================
# Step 6: pairs
# =============================================================================
!python scripts/make_pairs.py --a outputs/generations/dpo_hh_hh.json --a_name dpo_hh --b outputs/generations/sft_hh.json --b_name sft --out outputs/eval/dpo_hh_vs_sft_pairs.jsonl
!python scripts/make_pairs.py --a outputs/generations/dpo_uf_alpaca.json --a_name dpo_uf --b outputs/generations/sft_alpaca.json --b_name sft --out outputs/eval/dpo_uf_vs_sft_pairs.jsonl

In [ ]:
# =============================================================================
# Step 7: quick quantitative look before judging: response length per model
#   (LLM judges are biased toward longer answers - RA2 Sec. 6.4 - so report lengths)
# =============================================================================
import json, statistics
for f in sorted(glob.glob("outputs/generations/*.json")):
    if "_sanity" in f or f.endswith("_decoding.json"): continue
    rows = json.load(open(f))
    print(f"{os.path.basename(f):28s} n={len(rows):3d}  mean words={statistics.mean(len(r['output'].split()) for r in rows):6.1f}")
!zip -q -r /kaggle/working/generation_outputs.zip outputs/generations outputs/eval

## Final step · Preserve the outputs  (PA2 Sec. 5.5)
1. Click **Save Version → Save & Run All (Commit)** (or, if you ran interactively, *Quick Save* keeps the current outputs).
2. After the version finishes, its `/kaggle/working` folder becomes an **Output** you can attach to the next notebook with **Add Input → Your Work → Notebooks**.